In [8]:
import numpy as np
from math import pi
from scipy.optimize import root

AU_KM = 1.495978707e8

# Orbital elements from HW
bodies = {
    "Earth": {
        "a": 1.4765067e8,
        "e": 9.1669995e-3,
        "i": 4.2422693e-3,
        "w": 6.64375167e1,
        "Omega": 1.4760836e1,
    },
    "Apophis": {
        "a": 1.3793939e8,
        "e": 1.9097084e-1,
        "i": 3.3356539,
        "w": 1.2919949e2,
        "Omega": 2.0381969e2,
    },
    "2024 YR4": {
        "a": 3.7680703e8,
        "e": 6.6164147e-1,
        "i": 3.4001497,
        "w": 1.3429905e2,
        "Omega": 2.7147904e2,
    },
    "3I/ATLAS": {
        "a": -3.9552667e7,
        "e": 6.1469268,
        "i": 1.7512507e2,
        "w": 1.2817255e2,
        "Omega": 3.2228906e2,
    },
}

def rotation_matrix(i_deg, w_deg, Omega_deg):
    i = np.deg2rad(i_deg)
    w = np.deg2rad(w_deg)
    O = np.deg2rad(Omega_deg)

    Rz_O = np.array([[np.cos(O), -np.sin(O), 0],[np.sin(O),  np.cos(O), 0],[0, 0, 1]])
    Rx_i = np.array([[1, 0, 0],[0, np.cos(i), -np.sin(i)],[0, np.sin(i),  np.cos(i)]])
    Rz_w = np.array([[np.cos(w), -np.sin(w), 0],[np.sin(w),  np.cos(w), 0],[0, 0, 1]])
    
    return Rz_O @ Rx_i @ Rz_w

def r_from_elements(body, f):
    a = body["a"]
    e = body["e"]
    
    R = rotation_matrix(body["i"], body["w"], body["Omega"])
   
    r_mag = a * (1 - e**2) / (1 + e * np.cos(f))
    r_pf = np.array([r_mag * np.cos(f), r_mag * np.sin(f), 0.0])
    
    return R @ r_pf

def d2_pair(f_vec, body1, body2):
    f1, f2 = f_vec
    
    r1 = r_from_elements(body1, f1)
    r2 = r_from_elements(body2, f2)
    
    return np.dot(r1 - r2, r1 - r2)

def grad_d2(f_vec, body1, body2, h=1e-5):
    f1, f2 = f_vec
    
    d2_p1 = d2_pair((f1 + h, f2), body1, body2)
    d2_m1 = d2_pair((f1 - h, f2), body1, body2)
    d2_p2 = d2_pair((f1, f2 + h), body1, body2)
    d2_m2 = d2_pair((f1, f2 - h), body1, body2)
    
    df1 = (d2_p1 - d2_m1) / (2*h)
    df2 = (d2_p2 - d2_m2) / (2*h)
    
    return np.array([df1, df2])

def solve_stationary_points(body1, body2, n_starts=12):
    sols = []
    for i in range(n_starts):
        for j in range(n_starts):
            x0 = np.array([2*pi*i/n_starts, 2*pi*j/n_starts])
            def F(x):
                return grad_d2(np.mod(x, 2*pi), body1, body2)
            res = root(F, x0)
            if res.success:
                f1, f2 = np.mod(res.x, 2*pi)
                sols.append((f1, f2))
    unique = []
    for f1, f2 in sols:
        if not any(np.hypot(f1-u1, f2-u2) < 1e-3 for u1, u2 in unique):
            unique.append((f1, f2))
    return unique

def moid_sitarski(body1, body2):
    candidates = solve_stationary_points(body1, body2)
    best = None
    best_d2 = np.inf

    for f1, f2 in candidates:
        d2 = d2_pair((f1, f2), body1, body2)
        if d2 < best_d2:
            best_d2 = d2
            best = (f1, f2)

    d_km = np.sqrt(best_d2)
    d_au = d_km / AU_KM
    return best[0], best[1], d_km, d_au

pairs = [
    ("Earth", "Apophis"),
    ("Earth", "2024 YR4"),
    ("Apophis", "2024 YR4"),
    ("Earth", "3I/ATLAS"),
]

for p in pairs:
    f1, f2, d_km, d_au = moid_sitarski(bodies[p[0]], bodies[p[1]])
    print(f"\n=== {p[0]}  vs  {p[1]} ===")
    print(f"f1 = {f1:.6f} rad   ({np.rad2deg(f1):.3f} deg)")
    print(f"f2 = {f2:.6f} rad   ({np.rad2deg(f2):.3f} deg)")
    print(f"MOID = {d_km:.3f} km   = {d_au:.9f} AU")


=== Earth  vs  Apophis ===
f1 = 2.233453 rad   (127.967 deg)
f2 = 4.121681 rad   (236.155 deg)
MOID = 847896.669 km   = 0.005667839 AU

=== Earth  vs  2024 YR4 ===
f1 = 0.208165 rad   (11.927 deg)
f2 = 0.826417 rad   (47.350 deg)
MOID = 264452.644 km   = 0.001767757 AU

=== Apophis  vs  2024 YR4 ===
f1 = 2.169393 rad   (124.297 deg)
f2 = 0.898949 rad   (51.506 deg)
MOID = 7530657.644 km   = 0.050339337 AU

=== Earth  vs  3I/ATLAS ===
f1 = 1.967502 rad   (112.730 deg)
f2 = 0.003817 rad   (0.219 deg)
MOID = 56610142.751 km   = 0.378415431 AU
